# Prepare the project splits from EB-NeRD small

The original training split is retained. Validation and test are whole-user
samples from the original SMALL validation split, targeting approximately
`train rows / 8` each. Validation uses seed 42; test uses seed 43 and excludes
validation users. Both samples retain their matching original validation histories.

The resulting ratio is approximately 80–10–10 by behaviour rows. Validation and
test share the original validation period and have no users in common with each
other; users may also appear in the earlier training period. Test labels are
reserved for evaluation after model and reranker settings are frozen.

The same 20,738-article SMALL catalogue is saved under train_val/ and test/.
The original training behaviours and histories are copied to train_val/train/.


In [1]:
from pathlib import Path

import json
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

COLAB_NOTEBOOKS = Path('/content/drive/MyDrive/Colab Notebooks')
DATA_ROOT = COLAB_NOTEBOOKS / 'BT4222 Shared' / 'ebnerd_small'
OUTPUT_DIR = COLAB_NOTEBOOKS / 'BT4222 Shared Final Folder'
VALIDATION_DIR = OUTPUT_DIR / 'train_val' / 'validation'
OUTPUT_PATH = next((p for p in (VALIDATION_DIR / 'behaviours.parquet',
                               VALIDATION_DIR / 'behaviors.parquet') if p.is_file()),
                   VALIDATION_DIR / 'behaviours.parquet')
OUTPUT_HISTORY_PATH = VALIDATION_DIR / 'history.parquet'

TRAIN_PATH = DATA_ROOT / 'train' / 'behaviors.parquet'
SOURCE_VALIDATION_PATH = DATA_ROOT / 'validation' / 'behaviors.parquet'
SOURCE_VALIDATION_HISTORY_PATH = DATA_ROOT / 'validation' / 'history.parquet'
for path in (TRAIN_PATH, SOURCE_VALIDATION_PATH, SOURCE_VALIDATION_HISTORY_PATH):
    if not path.is_file():
        raise FileNotFoundError(f'Source file not found: {path}')

Mounted at /content/drive


## 1. Load and count source behaviors

In [2]:
train_behaviors = pd.read_parquet(TRAIN_PATH)
source_validation_behaviors = pd.read_parquet(SOURCE_VALIDATION_PATH)

train_rows = len(train_behaviors)
source_validation_rows = len(source_validation_behaviors)
source_total_rows = train_rows + source_validation_rows
if source_total_rows == 0:
    raise ValueError('Both source behavior files are empty.')

print(f'Train behaviors: {train_rows:,} rows; {train_behaviors["impression_id"].nunique():,} unique impressions')
print(f'Source validation behaviors: {source_validation_rows:,} rows; {source_validation_behaviors["impression_id"].nunique():,} unique impressions')
print(f'Combined source: {source_total_rows:,} rows')
print(f'Train share of combined source: {train_rows / source_total_rows:.2%}')

Train behaviors: 232,887 rows; 232,887 unique impressions
Source validation behaviors: 244,647 rows; 244,647 unique impressions
Combined source: 477,534 rows
Train share of combined source: 48.77%


## 2. Select validation users

Shuffle the source validation users with a fixed seed, then choose the cutoff whose cumulative behavior count is closest to the 10% target. This keeps every selected user's impressions together.

In [3]:
if train_rows == 0:
    raise ValueError('Train behaviors is empty; cannot size validation.')

implied_total_rows = train_rows / 0.80
target_validation_rows = round(implied_total_rows * 0.10)
if target_validation_rows > source_validation_rows:
    raise ValueError(
        f'Need {target_validation_rows:,} validation rows, but the source has '
        f'only {source_validation_rows:,}.'
    )

user_counts = source_validation_behaviors.groupby('user_id').size()
if user_counts.empty:
    raise ValueError('Source validation behaviors has no users.')
rng = np.random.default_rng(42)
shuffled_user_counts = user_counts.iloc[rng.permutation(len(user_counts))]
cumulative_rows = shuffled_user_counts.cumsum().to_numpy()
selected_user_count = int(np.argmin(np.abs(cumulative_rows - target_validation_rows))) + 1
selected_user_ids = shuffled_user_counts.index[:selected_user_count]
validation_mask = source_validation_behaviors['user_id'].isin(selected_user_ids)
validation_behaviors = source_validation_behaviors.loc[validation_mask].reset_index(drop=True)
remaining_test_behaviors = source_validation_behaviors.loc[~validation_mask]
shared_train_users = selected_user_ids.intersection(train_behaviors['user_id'].unique())

print(f'Implied total (train = 80%): {implied_total_rows:,.0f} rows')
print(f'Validation target (10%): {target_validation_rows:,} rows')
print(f'Selected: {selected_user_count:,} users and {len(validation_behaviors):,} rows')
print(f'Actual validation share of implied total: {len(validation_behaviors) / implied_total_rows:.2%}')
print(
    f'Unselected source validation: {len(remaining_test_behaviors):,} rows, '
    f'{remaining_test_behaviors["user_id"].nunique():,} users'
)
print(f'Selected users also present in source train: {len(shared_train_users):,}')

Implied total (train = 80%): 291,109 rows
Validation target (10%): 29,111 rows
Selected: 1,719 users and 29,110 rows
Actual validation share of implied total: 10.00%
Unselected source validation: 215,537 rows, 13,623 users
Selected users also present in source train: 1,317


## 3. Filter history, then save and inspect validation

A history row belongs to one user. Keep each selected user's full fixed history arrays; remove history rows for users outside the selected validation set.

In [4]:
source_validation_history = pd.read_parquet(SOURCE_VALIDATION_HISTORY_PATH)
validation_history = (
    source_validation_history.loc[
        source_validation_history['user_id'].isin(selected_user_ids)
    ].reset_index(drop=True)
)
assert validation_history['user_id'].isin(validation_behaviors['user_id']).all()
missing_history_users = selected_user_ids.difference(validation_history['user_id'].unique())

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
validation_behaviors.to_parquet(OUTPUT_PATH, index=False)
validation_history.to_parquet(OUTPUT_HISTORY_PATH, index=False)

print(f'Saved behaviors to: {OUTPUT_PATH}')
print(f'Saved history to: {OUTPUT_HISTORY_PATH}')
print(f'Validation shape: {validation_behaviors.shape[0]:,} rows × {validation_behaviors.shape[1]:,} columns')
print(f'Unique impressions: {validation_behaviors["impression_id"].nunique():,}')
print(f'Unique users: {validation_behaviors["user_id"].nunique():,}')
print(f'History rows / users: {len(validation_history):,} / {validation_history["user_id"].nunique():,}')
print(f'Selected behavior users without a source history row: {len(missing_history_users):,}')
print(f'Behavior file size: {OUTPUT_PATH.stat().st_size / (1024 ** 2):,.2f} MiB')
print(f'History file size: {OUTPUT_HISTORY_PATH.stat().st_size / (1024 ** 2):,.2f} MiB')

Saved behaviors to: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/train_val/validation/behaviours.parquet
Saved history to: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/train_val/validation/history.parquet
Validation shape: 29,110 rows × 17 columns
Unique impressions: 29,110
Unique users: 1,719
History rows / users: 1,719 / 1,719
Selected behavior users without a source history row: 0
Behavior file size: 1.28 MiB
History file size: 2.86 MiB


## 4. Reopen the saved files and describe them

Run this in Colab after the Drive setup cell. It reads `OUTPUT_PATH` and `OUTPUT_HISTORY_PATH` from the final Drive folder, so the statistics describe the saved Parquet files. If you renamed either file in Drive, change its path in the setup cell first.

In [5]:
from IPython.display import display

for path in (OUTPUT_PATH, OUTPUT_HISTORY_PATH):
    if not path.is_file():
        raise FileNotFoundError(f'Saved validation file not found: {path}')

saved_behaviors = pd.read_parquet(OUTPUT_PATH)
saved_history = pd.read_parquet(OUTPUT_HISTORY_PATH)

def list_size(value):
    return len(value) if isinstance(value, (list, tuple, np.ndarray)) else 0

inview_counts = saved_behaviors['article_ids_inview'].map(list_size)
clicked_counts = saved_behaviors['article_ids_clicked'].map(list_size)
history_lengths = saved_history['article_id_fixed'].map(list_size)
behavior_users = pd.Index(saved_behaviors['user_id'].unique())
history_users = pd.Index(saved_history['user_id'].unique())

print('Saved validation files')
display(pd.DataFrame([
    {'file': OUTPUT_PATH.name, 'rows': len(saved_behaviors),
     'columns': saved_behaviors.shape[1],
     'size_MiB': round(OUTPUT_PATH.stat().st_size / 1024**2, 2)},
    {'file': OUTPUT_HISTORY_PATH.name, 'rows': len(saved_history),
     'columns': saved_history.shape[1],
     'size_MiB': round(OUTPUT_HISTORY_PATH.stat().st_size / 1024**2, 2)},
]))

print('Behavior summary')
display(pd.DataFrame([
    {'impressions': saved_behaviors['impression_id'].nunique(),
     'users': len(behavior_users),
     'first_impression': saved_behaviors['impression_time'].min(),
     'last_impression': saved_behaviors['impression_time'].max(),
     'candidate_exposures': int(inview_counts.sum()),
     'clicked_articles': int(clicked_counts.sum()),
     'impressions_with_click': int(clicked_counts.gt(0).sum()),
     'candidate_CTR': (clicked_counts.sum() / inview_counts.sum()
                       if inview_counts.sum() else float('nan'))}
]))
print('Candidates and clicks per impression')
display(pd.DataFrame({'candidates': inview_counts, 'clicks': clicked_counts})
        .describe(percentiles=[0.25, 0.5, 0.75, 0.9]).T)

print('History summary')
display(pd.DataFrame([
    {'history_rows': len(saved_history),
     'history_users': len(history_users),
     'duplicate_history_user_rows': len(saved_history) - len(history_users),
     'behavior_users_without_history': len(behavior_users.difference(history_users)),
     'history_users_without_behaviors': len(history_users.difference(behavior_users))}
]))
print('Fixed history events per user')
display(history_lengths.describe(percentiles=[0.25, 0.5, 0.75, 0.9]).to_frame().T)

print('Missing values by behavior column')
display(saved_behaviors.isna().sum().rename('missing_rows').to_frame()
        .sort_values('missing_rows', ascending=False))

Saved validation files


,file,rows,columns,size_MiB
0,behaviours.parquet,29110,17,1.28
1,history.parquet,1719,5,2.86


Behavior summary


,impressions,users,first_impression,last_impression,candidate_exposures,clicked_articles,impressions_with_click,candidate_CTR
0,29110,1719,2023-05-25 07:00:15,2023-06-01 06:59:46,344666,29322,29110,0.085074


Candidates and clicks per impression


,count,mean,std,min,25%,50%,75%,90%,max
candidates,29110.0,11.840124,9.141413,5.0,6.0,9.0,14.0,22.0,91.0
clicks,29110.0,1.007283,0.104595,1.0,1.0,1.0,1.0,1.0,5.0


... (2 outputs left out)

Fixed history events per user


,count,mean,std,min,25%,50%,75%,90%,max
article_id_fixed,1719.0,146.926702,163.964076,5.0,34.0,83.0,202.0,372.2,960.0


Missing values by behavior column


,missing_rows
postcode,28597
age,28423
gender,27477
scroll_percentage,21089
article_id,21004
next_scroll_percentage,3468
next_read_time,765
impression_id,0
impression_time,0
user_id,0


## 5. Sample and save the test users

Select whole users from the remaining original SMALL validation data, excluding validation users. Target the same 10% row count with seed 43 and save their behaviours and matching histories under test/. Labels are retained for final evaluation.


In [6]:
TEST_OUTPUT_DIR = OUTPUT_DIR / 'test'
TEST_OUTPUT_BEHAVIORS_PATH = TEST_OUTPUT_DIR / 'behaviors.parquet'
TEST_OUTPUT_HISTORY_PATH = TEST_OUTPUT_DIR / 'history.parquet'
source_test_behaviors = remaining_test_behaviors
source_test_history = source_validation_history
assert 'article_ids_clicked' in source_test_behaviors
assert source_test_behaviors['article_ids_clicked'].notna().all()
print(f'Available holdout source: {len(source_test_behaviors):,} behaviours from unused validation users')

Available holdout source: 215,537 behaviours from unused validation users


In [7]:
target_test_rows = round(train_rows / 8)
eligible_test_behaviors = source_test_behaviors.loc[
    ~source_test_behaviors['user_id'].isin(selected_user_ids)
]
test_user_counts = eligible_test_behaviors.groupby('user_id').size()
if target_test_rows > len(eligible_test_behaviors) or test_user_counts.empty:
    raise ValueError(
        f'Need about {target_test_rows:,} test rows from users outside validation; '
        f'only {len(eligible_test_behaviors):,} eligible rows are available.'
    )

test_rng = np.random.default_rng(43)
shuffled_test_user_counts = test_user_counts.iloc[
    test_rng.permutation(len(test_user_counts))
]
test_cumulative_rows = shuffled_test_user_counts.cumsum().to_numpy()
selected_test_user_count = int(
    np.argmin(np.abs(test_cumulative_rows - target_test_rows))
) + 1
selected_test_user_ids = shuffled_test_user_counts.index[:selected_test_user_count]
test_behaviors = source_test_behaviors.loc[
    source_test_behaviors['user_id'].isin(selected_test_user_ids)
].reset_index(drop=True)
test_history = source_test_history.loc[
    source_test_history['user_id'].isin(selected_test_user_ids)
].reset_index(drop=True)
test_impression_count = test_behaviors[['user_id', 'impression_id']].drop_duplicates().shape[0]
reused_test_impression_ids = (
    test_behaviors['impression_id'].notna().sum()
    - test_behaviors['impression_id'].nunique()
)
repeated_test_user_impression_pairs = test_behaviors.duplicated(
    ['user_id', 'impression_id']
).sum()

assert selected_test_user_ids.intersection(selected_user_ids).empty
assert test_history['user_id'].isin(test_behaviors['user_id']).all()
test_users_without_history = selected_test_user_ids.difference(
    test_history['user_id'].unique()
)
test_users_in_train = selected_test_user_ids.intersection(
    train_behaviors['user_id'].unique()
)

assert not test_behaviors.duplicated(['user_id', 'impression_id']).any()
assert test_users_without_history.empty
TEST_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
test_behaviors.to_parquet(TEST_OUTPUT_BEHAVIORS_PATH, index=False)
test_history.to_parquet(TEST_OUTPUT_HISTORY_PATH, index=False)

# A compact membership manifest identifies this labelled project dataset.
manifest = {
    "protocol": "ebnerd_small_labelled_user_holdout_v1",
    "ratio_basis": "behaviour rows; train retained as 80%, each evaluation split targets train/8",
    "source_validation": str(SOURCE_VALIDATION_PATH),
    "source_history": str(SOURCE_VALIDATION_HISTORY_PATH),
    "validation_membership": "seed-42 whole-user sample of original SMALL validation",
    "holdout_selection": {"seed": 43, "unit": "whole user", "exclude": "all project validation users"},
    "validation_user_ids": sorted(map(int, selected_user_ids)),
    "holdout_user_ids": sorted(map(int, selected_test_user_ids)),
    "validation_holdout_user_overlap": 0,
    "evaluation_period": "validation and holdout share the original validation week; not a later-week temporal test",
    "splits": {
        "train": {"rows": int(train_rows), "labelled": True, "role": "fit models and preprocessing"},
        "validation": {"rows": int(len(validation_behaviors)), "labelled": True,
                       "role": "tune models, select checkpoints and tune the reranker"},
        "test": {"rows": int(len(test_behaviors)), "labelled": True,
                 "role": "final evaluation after the whole pipeline is frozen",
                 "behaviors": "test/behaviors.parquet", "history": "test/history.parquet"},
    },
    "catalogue_policy": "published_at_or_before_request_v1",
}
(OUTPUT_DIR / 'split_manifest.json').write_text(json.dumps(manifest, indent=2) + '\n')
print(f'Saved holdout: {len(test_behaviors):,} behaviours, {len(selected_test_user_ids):,} users')
print('Holdout labels retained; no holdout click metrics calculated.')
print('Behaviours:', TEST_OUTPUT_BEHAVIORS_PATH)
print('Histories:', TEST_OUTPUT_HISTORY_PATH)
print('Split manifest:', OUTPUT_DIR / 'split_manifest.json')

Saved holdout: 29,142 behaviours, 1,757 users
Holdout labels retained; no holdout click metrics calculated.
Behaviours: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/test/behaviors.parquet
Histories: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/test/history.parquet
Split manifest: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/split_manifest.json


## 6. Save the shared SMALL catalogue and original training files

All splits use the same SMALL article snapshot. If an existing catalogue changes, its materialised union cache is archived before the files are replaced. Publish the complete article embeddings with `publish_article_embeddings.py` as described in the preprocessing README.

In [ ]:
from datetime import datetime, timezone
import hashlib
import shutil

# Retain the original SMALL training data and use its catalogue for every split.
copies = [
    (DATA_ROOT / 'train' / 'behaviors.parquet', OUTPUT_DIR / 'train_val/train/behaviors.parquet'),
    (DATA_ROOT / 'train' / 'history.parquet', OUTPUT_DIR / 'train_val/train/history.parquet'),
    (DATA_ROOT / 'articles.parquet', OUTPUT_DIR / 'train_val/articles.parquet'),
    (DATA_ROOT / 'articles.parquet', OUTPUT_DIR / 'test/articles.parquet'),
]
for source, _ in copies:
    if not source.is_file():
        raise FileNotFoundError(f'Source file not found: {source}')

def file_hash(path):
    with path.open('rb') as source:
        return hashlib.file_digest(source, 'sha256').hexdigest()

catalogue_changed = any(
    target.is_file() and file_hash(source) != file_hash(target)
    for source, target in copies if source.name == 'articles.parquet'
)
if catalogue_changed:
    archive = OUTPUT_DIR / 'archive' / datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_before_small_catalogue') / 'catalogue'
    for name in ('article_union.parquet', 'article_union.parquet.manifest.json'):
        cache = OUTPUT_DIR / 'catalogue' / name
        if cache.is_file():
            archive.mkdir(parents=True, exist_ok=True)
            shutil.move(str(cache), str(archive / name))

for source, target in copies:
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, target)
    print('Saved:', target)
